# 04_SES_Feature_Engineering
Build Skill Extinction Score (SES) dataset from LinkedIn + StackOverflow data

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.preprocessing import MinMaxScaler


## Load Datasets

In [2]:
job_skills = pd.read_csv('job_skills.csv')
linkedin = pd.read_csv('linkedin_job_postings.csv')

job_skills_map = pd.read_csv('job_skills_1.csv')
skills_map = pd.read_csv('skills_1.csv')

job_industries = pd.read_csv('job_industries.csv')
industries = pd.read_csv('industries.csv')

salaries = pd.read_csv('salaries.csv')
stack = pd.read_csv('survey_results_public.csv', low_memory=False)


## LinkedIn Demand Score

In [3]:
all_skills=[]

for row in job_skills['job_skills'].dropna():
    all_skills.extend([x.strip() for x in str(row).split(',')])

demand_df=(pd.Series(all_skills)
           .value_counts()
           .reset_index())

demand_df.columns=['skill','demand_count']
demand_df.head()


,skill,demand_count
0,Communication,368293
1,Teamwork,226266
2,Leadership,184341
3,Customer service,166209
4,Communication skills,116260


## Developer Interest Score (StackOverflow)

In [4]:
def extract_counts(series):
    counter=Counter()
    for row in series.dropna():
        for item in str(row).split(';'):
            counter[item.strip()] += 1
    return counter

interest_counter=Counter()

for col in [
    'LanguageWantToWorkWith',
    'DatabaseWantToWorkWith',
    'PlatformWantToWorkWith',
    'WebframeWantToWorkWith',
    'ToolsTechWantToWorkWith',
    'MiscTechWantToWorkWith'
]:
    interest_counter.update(extract_counts(stack[col]))

interest_df=pd.DataFrame(
    interest_counter.items(),
    columns=['skill','developer_interest']
)

interest_df.sort_values(
    'developer_interest',
    ascending=False
).head()


,skill,developer_interest
146,Docker,26262
6,Python,25047
49,PostgreSQL,24005
4,JavaScript,23774
10,SQL,22400


## Salary Score

In [5]:
salary_score = salaries[['max_salary','min_salary']].copy()

salary_score['salary_score']=(
    salary_score['max_salary'].fillna(0) +
    salary_score['min_salary'].fillna(0)
)/2

salary_score['salary_score'].describe()


count    4.078500e+04
mean     6.712628e+04
std      5.131228e+05
min      0.000000e+00
25%      2.162500e+01
50%      5.380000e+04
75%      1.100750e+05
max      1.025000e+08
Name: salary_score, dtype: float64

## Industry Adoption Score

In [6]:
skill_industry = (
    job_skills_map
    .groupby('skill_abr')
    .size()
    .reset_index(name='industry_adoption')
)

skill_industry = skill_industry.merge(
    skills_map,
    on='skill_abr',
    how='left'
)

skill_industry = skill_industry[[
    'skill_name',
    'industry_adoption'
]]

skill_industry.columns=['skill','industry_adoption']
skill_industry.head()


,skill,industry_adoption
0,Accounting/Auditing,5461
1,Administrative,4860
2,Advertising,681
3,Analyst,3858
4,Art/Creative,1664


## Merge SES Features

In [7]:
ses_df = demand_df.merge(
    interest_df,
    on='skill',
    how='outer'
)

ses_df = ses_df.merge(
    skill_industry,
    on='skill',
    how='left'
)

ses_df = ses_df.fillna(0)

ses_df.head()


,skill,demand_count,developer_interest,industry_adoption
0,,14.0,0.0,0.0
1,"""20 Tools of Process Control""",1.0,0.0,0.0
2,"""2nd Approval"" Opportunities",1.0,0.0,0.0
3,"""6 days on and 2 days off guaranteed"" schedule",1.0,0.0,0.0
4,"""A school"" or ""C school""",1.0,0.0,0.0


## Normalize Features

In [8]:
scaler = MinMaxScaler()

for col in [
    'demand_count',
    'developer_interest',
    'industry_adoption'
]:
    if col in ses_df.columns:
        ses_df[col] = scaler.fit_transform(
            ses_df[[col]]
        )


## Calculate SES

In [9]:
ses_df['SES'] = (
    0.35 * ses_df['demand_count'] +
    0.25 * ses_df['developer_interest'] +
    0.15 * ses_df['industry_adoption']
)

ses_df['SES'] = ses_df['SES'] * 10

ses_df[['skill','SES']].head()


,skill,SES
0,,0.000133
1,"""20 Tools of Process Control""",0.000010
2,"""2nd Approval"" Opportunities",0.000010
3,"""6 days on and 2 days off guaranteed"" schedule",0.000010
4,"""A school"" or ""C school""",0.000010


## Classification

In [10]:
def classify(score):
    if score < 3:
        return 'High Risk'
    elif score < 6:
        return 'Stable'
    elif score < 8:
        return 'Growing'
    return 'Future-Proof'

ses_df['classification'] = ses_df['SES'].apply(classify)


## Top Skills by SES

In [11]:
ses_df.sort_values(
    'SES',
    ascending=False
)[['skill','SES','classification']].head(50)


,skill,SES,classification
746815,Communication,3.500000,Stable
2490369,Python,2.622862,High Risk
1032730,Docker,2.559424,High Risk
1681466,JavaScript,2.342727,High Risk
2663353,SQL,2.341640,High Risk
2341361,PostgreSQL,2.300057,High Risk
2685369,Sales,2.171422,High Risk
2971512,Teamwork,2.150274,High Risk
1460024,HTML/CSS,1.980339,High Risk
3079741,TypeScript,1.948615,High Risk


## Export Results

In [12]:
ses_df.to_csv(
    'master_skill_features.csv',
    index=False
)

ses_df.sort_values(
    'SES',
    ascending=False
).to_csv(
    'ses_rankings.csv',
    index=False
)

print('Files exported successfully')


Files exported successfully
